# 04 Replay study: ICBHI audio played by a speaker in the lung phantom and recorded by an electret and a phone microphone

Chain: ICBHI audio -> speaker inside the lung phantom -> electret / phone microphone. 20 ICBHI recordings were played this way and recorded by the two capture devices. Each take is aligned to its original,
measured (delay, drift, SNR, coherence, log-spectral distance, transfer function) and passed through signal-processing
variants before the frozen models (distilled `kd_g4_k1` = AuscultTrust, and the gain6 anchor), seeds 0-4. Agreement is the
flip rate against the same model's prediction on the original audio; Score uses the ICBHI cycle labels.

Computed by `scripts/replay_analysis.py` -> `outputs/replay/phantom-replay-20261028/`. Patients 103, 105, 106, 107 are in
the official training split (their original audio was seen in training); 101, 102, 104 are test patients. The automatic
variant choice is made on the training-split takes and evaluated on the test-split takes.

In [ ]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))          # notebooks/common.py
sys.path.insert(0, str(Path.cwd().parent))   # repository root for src.*
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from common import *
pd.set_option('display.precision', 2); pd.set_option('display.width', 160)
FIG, TAB = out_dirs('replay')
RP = REPO / 'outputs/replay/phantom-replay-20261028'
takes = pd.read_csv(RP / 'takes.csv')
summ = pd.read_csv(RP / 'summary.csv')
cyc = pd.read_parquet(RP / 'cycles.parquet') if (RP / 'cycles.parquet').exists() else pd.read_csv(RP / 'cycles.csv.gz')
sel = json.loads((RP / 'selection.json').read_text())
tf = np.load(RP / 'transfer_functions.npz')
VARIANTS = ['raw', 'raw_sc', 'raw_sc_rec', 'raw_scd', 'bp_sc', 'dn_sc', 'dn_scd', 'dn_sc_tta', 'tdn_sc', 'tdn_scd', 'eq', 'dn_eq', 'dn_eq12', 'tdn_eq']
takes.head()
check_artifacts()


## 1. Signal quality of the takes

In [ ]:
q = takes.groupby('capture')[['lag_sec', 'gcc_peak_sd', 'drift_ppm', 'snr_db', 'coherence', 'lsd_db', 'rms_ratio_db']].agg(['mean', 'std', 'min', 'max'])
q.to_csv(TAB / 'take_quality.csv'); display(q.round(2))
fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for i, m in enumerate(['snr_db', 'coherence', 'lsd_db', 'gcc_peak_sd']):
    sns.boxplot(data=takes, x='capture', y=m, ax=ax[i]); sns.stripplot(data=takes, x='capture', y=m, hue='device', ax=ax[i], size=5)
    ax[i].set_title(m); ax[i].get_legend().remove() if i < 3 else ax[i].legend(fontsize=7)
savefig(fig, FIG, 'take_quality')

## 2. Capture transfer functions |H(f)| = sqrt(PSD_take / PSD_original)

In [ ]:
f = tf['freqs']
fig, ax = plt.subplots(1, 2, figsize=(15, 4), sharey=True)
for i, c in enumerate(sorted(takes.capture.unique())):
    H = np.array([20 * np.log10(np.maximum(tf[k], 1e-6)) for k in tf.files if k.startswith(c + '__')])
    H = H - np.median(H[:, (f > 200) & (f < 1000)], axis=1, keepdims=True)
    for h in H: ax[i].semilogx(f[1:], h[1:], color='grey', alpha=0.25, lw=0.6)
    ax[i].semilogx(f[1:], np.median(H, 0)[1:], color='C3', lw=2, label='median')
    ax[i].fill_between(f[1:], *np.percentile(H, [25, 75], 0)[:, 1:], color='C3', alpha=0.2, label='IQR')
    ax[i].axvspan(50, 2000, color='C0', alpha=0.05); ax[i].set_title(f'{c}: relative gain (dB, 0 dB at 200-1000 Hz)'); ax[i].set_xlabel('Hz')
ax[0].set_ylim(-40, 25); ax[0].legend()
savefig(fig, FIG, 'transfer_functions')

## 3. Spectrogram: original vs takes vs processed (one test recording)

In [ ]:
import librosa, librosa.display
from src.processing.audio import load_wav
stem = '104_1b1_Al_sc_Litt3200'
items = [('original', load_wav(DATA / 'raw/icbhi/ICBHI_final_database' / f'{stem}.wav'))]
for c in ['phone', 'electret']:
    for v in ['raw', 'tn', 'tdn', 'tdn_eq']:
        p = RP / 'audio' / c / v / f'{stem}.wav'
        if p.exists(): items.append((f'{c} / {v}', load_wav(p)))
fig, ax = plt.subplots(len(items), 1, figsize=(14, 2.1 * len(items)), sharex=True)
for a, (n, x) in zip(ax, items):
    m = librosa.power_to_db(librosa.feature.melspectrogram(y=x, sr=16000, n_fft=1024, hop_length=256, n_mels=96, fmax=4000), ref=np.max)
    librosa.display.specshow(m, sr=16000, hop_length=256, y_axis='mel', fmax=4000, ax=a, vmin=-70); a.set_ylabel(n, fontsize=8)
savefig(fig, FIG, f'spectrogram_{stem}')

## 4. Agreement with the original prediction and Score, per processing variant

In [ ]:
S = summ[summ.capture != 'original']
for split in ['test', 'train']:
    t = S[S.split == split].pivot_table(index=['model', 'variant'], columns='capture', values=['flip_mean', 'flip2_mean', 'score_mean'])
    t.to_csv(TAB / f'variants_{split}.csv'); print(f'== {split}-split takes'); display(t.round(3))
o = summ[summ.capture == 'original'].pivot_table(index=['model', 'variant'], columns='split', values='score_mean'); print('== original audio Score'); display(o.round(2))

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(18, 8), sharey='row')
for j, split in enumerate(['test', 'train']):
    g = S[(S.split == split) & (S.model == 'kd_g4_k1')]
    sns.barplot(data=g, x='variant', y='flip_mean', hue='capture', order=VARIANTS, ax=ax[0, j]); ax[0, j].set_title(f'kd_g4_k1 4-class flip rate vs original ({split} patients)')
    sns.barplot(data=g, x='variant', y='score_mean', hue='capture', order=VARIANTS, ax=ax[1, j]); ax[1, j].set_title(f'kd_g4_k1 Score ({split} patients)')
    for a in ax[:, j]: a.tick_params(axis='x', rotation=30)
    o = summ[(summ.capture == 'original') & (summ.variant == 'original') & (summ.split == split) & (summ.model == 'kd_g4_k1')].score_mean
    ax[1, j].axhline(o.iloc[0], color='k', ls='--', lw=1, label='original audio'); ax[1, j].legend(fontsize=7)
savefig(fig, FIG, 'variants_kd_g4_k1')

### Seed ensemble and normal-vs-abnormal agreement

`flip2` = share of cycles whose normal/abnormal decision changes against the original; `_ens5` = mean softmax over seeds.

In [ ]:
t = S[(S.split == 'test') & S.model.isin(['kd_g4_k1', 'kd_g4_k1_ens5', 'anchor', 'anchor_ens5'])]
t = t.pivot_table(index='variant', columns=['model', 'capture'], values='flip2_mean').reindex(VARIANTS)
t.to_csv(TAB / 'flip2_test.csv'); t.round(3)

## 5. Automatic choice (selected on training-split takes, evaluated on test-split takes)

In [ ]:
ch = pd.DataFrame(sel['choice']).T
ch.to_csv(TAB / 'auto_choice.csv'); ch

## 6. Per-take flip rate vs signal quality (kd_g4_k1, raw_sc)

In [ ]:
k = cyc[(cyc.model == 'kd_g4_k1') & (cyc.variant == 'raw_sc')].groupby(['capture', 'stem']).flip.mean().rename('flip').reset_index()
k = k.merge(takes, on=['capture', 'stem'])
fig, ax = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for i, m in enumerate(['snr_db', 'coherence', 'lsd_db']):
    sns.scatterplot(data=k, x=m, y='flip', hue='capture', style='split', ax=ax[i], s=50)
    ax[i].set_title(f'flip vs {m} (Spearman {k[[m, "flip"]].corr("spearman").iloc[0, 1]:.2f})')
savefig(fig, FIG, 'flip_vs_quality')

## 7. Class-level change: confusion of original vs replay predictions (kd_g4_k1, chosen variant)

In [ ]:
from sklearn.metrics import confusion_matrix
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for i, c in enumerate(['phone', 'electret']):
    v = sel['choice'][f'kd_g4_k1/{c}']['chosen']
    g = cyc[(cyc.model == 'kd_g4_k1') & (cyc.capture == c) & (cyc.variant == v)]
    cm = confusion_matrix(g.pred_orig, g.pred, labels=range(4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=CLASSES, yticklabels=CLASSES, ax=ax[i])
    ax[i].set_title(f'{c} / {v}'); ax[i].set_xlabel('replay prediction'); ax[i].set_ylabel('original prediction')
savefig(fig, FIG, 'confusion_original_vs_replay')